# IN16: Capstone Solution
## Walmart Retail Assistant -- Production AI System Design

This notebook is the reference solution for [IN16_Capstone_Problem_Statement.ipynb](IN16_Capstone_Problem_Statement.ipynb). It completes all six Architecture Review Board (ARB) components:

| # | Component | Implemented in |
|---|---|---|
| 1 | Architecture choice | Weighted decision matrix and ADR, saved to `outputs/capstone_adr.txt` |
| 2 | Agent strategy | `WalmartRetailAgent`: a classifier, tool dispatch, retrieval, then a call to the LLM it picked |
| 3 | Evaluation strategy | A 10-metric scorecard on the golden dataset, scored by an LLM-as-judge, saved to `outputs/capstone_evaluation_scorecard.txt` |
| 4 | Cost model | Monthly spend under three scenarios, checked against the $1,500 ceiling |
| 5 | Deployment model | CI/CD and rollback plan, saved to `outputs/capstone_deployment_model.txt` |
| 6 | Risk mitigation | A register of six risks, saved to `outputs/capstone_risk_register.txt` |

**Requirements:** `OPENAI_API_KEY` in a `.env` file. The notebook uses the provided mock retriever, so it does not need Pinecone.

Generated files go into `outputs/` so they don't overwrite the reference docs in `docs/`.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'openai', 'python-dotenv', 'tiktoken'], check=True)
print('Packages ready.')

In [ ]:
import os, re, json, time, uuid, hashlib, statistics
from pathlib import Path
from datetime import datetime, timezone
from dotenv import load_dotenv
from openai import OpenAI
import tiktoken

load_dotenv(override=True)
if not os.getenv('OPENAI_API_KEY'):
    raise EnvironmentError('OPENAI_API_KEY not found. Add it to your .env file.')
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

OUT_DIR = Path('outputs')
OUT_DIR.mkdir(exist_ok=True)
print('Environment loaded. Outputs ->', OUT_DIR.resolve())

## Provided: Mock Knowledge Base and Tool Definitions
Copied unchanged from the problem statement.

In [ ]:
# Mock knowledge base (use this if Pinecone is unavailable)
KNOWLEDGE_BASE = [
    {'id': 'P001', 'category': 'price',  'text': 'Great Value Whole Milk 1 gallon is priced at $3.98. Located in Aisle 12, Dairy section. In stock: 47 units.'},
    {'id': 'P002', 'category': 'price',  'text': 'Great Value 2% Milk 1 gallon is priced at $3.78. Located in Aisle 12, Dairy section. In stock: 32 units.'},
    {'id': 'P003', 'category': 'price',  'text': 'Tide Original Laundry Detergent 92 oz is $11.97 (13 cents/oz). Aisle 7, Cleaning supplies.'},
    {'id': 'P004', 'category': 'price',  'text': 'Great Value Laundry Detergent 150 oz is $8.97 (6 cents/oz). Aisle 7, Cleaning supplies.'},
    {'id': 'O001', 'category': 'order',  'text': 'Order ORD-78901: shipped via FedEx, tracking FX123456, estimated delivery July 3 2026.'},
    {'id': 'O002', 'category': 'order',  'text': 'Order ORD-45621: processing, expected to ship within 2 business days.'},
    {'id': 'R001', 'category': 'policy', 'text': 'Electronics return policy: 30 days with receipt and original packaging. No exceptions.'},
    {'id': 'R002', 'category': 'policy', 'text': 'General return policy: 90 days with or without receipt. Without receipt: valid photo ID required, refund as store credit.'},
    {'id': 'H001', 'category': 'hours',  'text': 'Most Walmart Supercenters open at 6:00 AM and close at 11:00 PM Monday through Saturday.'},
    {'id': 'H002', 'category': 'hours',  'text': 'Sunday hours: 7:00 AM to 10:00 PM. Walmart stores are closed on Thanksgiving Day.'},
]

def mock_retrieve(query: str, k: int = 3) -> list:
    """Simple keyword-based mock retriever."""
    q = query.lower()
    scored = []
    for doc in KNOWLEDGE_BASE:
        score = sum(1 for word in q.split() if word in doc['text'].lower())
        if score > 0:
            scored.append((score, doc))
    scored.sort(key=lambda x: -x[0])
    return [doc for _, doc in scored[:k]]

# Tool stubs -- these simulate real tool calls
def price_lookup(product_name: str) -> dict:
    results = [d for d in KNOWLEDGE_BASE if d['category'] == 'price' and
               product_name.lower() in d['text'].lower()]
    return {'found': len(results) > 0, 'results': results}

def order_status(order_id: str) -> dict:
    results = [d for d in KNOWLEDGE_BASE if d['category'] == 'order' and
               order_id in d['text']]
    return {'found': len(results) > 0, 'results': results}

def policy_search(topic: str) -> dict:
    results = [d for d in KNOWLEDGE_BASE if d['category'] == 'policy' and
               any(w in d['text'].lower() for w in topic.lower().split())]
    return {'found': len(results) > 0, 'results': results}

def store_hours(day: str = '') -> dict:
    results = [d for d in KNOWLEDGE_BASE if d['category'] == 'hours']
    return {'found': True, 'results': results}

TOOLS = {
    'price_lookup':  price_lookup,
    'order_status':  order_status,
    'policy_search': policy_search,
    'store_hours':   store_hours,
}
print('Knowledge base and tools ready.')
print(f'Documents: {len(KNOWLEDGE_BASE)} | Tools: {list(TOOLS.keys())}')

---
## Task 1: Architecture Choice -- Decision Matrix and ADR

Each option gets a score from 1 to 5 on four weighted criteria: cost 30%, latency 25%, quality 30% and maintainability 15%. The option with the highest weighted total wins.

**ARB: Why an agent and not a deterministic workflow?** About 1 in 5 queries are `multi_step`, such as comparing products or checking stock and aisle together. Those need tools combined at runtime. A deterministic workflow would need a hand-coded branch for every combination of intents. The agent keeps single-intent queries on a cheap, deterministic path: a rule classifier, one tool, and `gpt-4o-mini`. It only pays for the stronger model on multi-intent queries.

In [ ]:
# ── 1a: Decision matrix ──────────────────────────────────────────────────
options = ['RAG + Agent', 'RAG + Workflow', 'Traditional Search']

criteria = [
    ('cost',            0.30),
    ('latency',         0.25),
    ('quality',         0.30),
    ('maintainability', 0.15),
]
assert abs(sum(w for _, w in criteria) - 1.0) < 1e-9, 'Weights must sum to 1.0'

scores = {
    # Agent: best quality on multi-intent queries; routing keeps cost moderate
    'RAG + Agent':        {'cost': 3, 'latency': 3, 'quality': 5, 'maintainability': 4},
    # Workflow: cheap and fast but rigid -- every intent combination is a new branch
    'RAG + Workflow':     {'cost': 4, 'latency': 4, 'quality': 3, 'maintainability': 3},
    # Search: no generation, cannot compose answers or compare products
    'Traditional Search': {'cost': 5, 'latency': 5, 'quality': 1, 'maintainability': 2},
}

def decision_matrix(options, criteria, scores):
    totals = {o: round(sum(scores[o][c] * w for c, w in criteria), 3) for o in options}
    header = f"{'Option':<22}" + ''.join(f'{c}({int(w*100)}%)'.rjust(20) for c, w in criteria) + 'WEIGHTED'.rjust(12)
    print(header)
    print('-' * len(header))
    for o in sorted(options, key=lambda x: -totals[x]):
        row = f'{o:<22}' + ''.join(str(scores[o][c]).rjust(20) for c, _ in criteria)
        print(row + f'{totals[o]:>12.2f}')
    winner = max(totals, key=totals.get)
    print(f'\nWinner: {winner} ({totals[winner]:.2f})')
    return winner, totals

winner, totals = decision_matrix(options, criteria, scores)

In [ ]:
# ── 1b: Architecture Decision Record ─────────────────────────────────────
def generate_adr(title, winner, totals, context, rationale, consequences, alternatives_rejected):
    date = datetime.now(timezone.utc).strftime('%Y-%m-%d')
    adr_id = 'ADR-' + hashlib.sha1(title.encode()).hexdigest()[:6].upper()
    lines = [
        f'{adr_id}: {title}',
        '=' * 70,
        f'Date:     {date}',
        'Status:   ACCEPTED',
        'Deciders: Lead AI Engineer, Architecture Review Board',
        '',
        'CONTEXT',
        '-------', context, '',
        'OPTIONS CONSIDERED (weighted score, max 5.00)',
        '---------------------------------------------',
        *[f'  - {o:<22} {s:.2f}' for o, s in sorted(totals.items(), key=lambda x: -x[1])],
        '',
        f'DECISION: {winner}',
        '--------', rationale, '',
        'ALTERNATIVES REJECTED',
        '---------------------',
        *[f'  - {k}: {v}' for k, v in alternatives_rejected.items()],
        '',
        'CONSEQUENCES',
        '------------',
        *[f'  {sign} {c}' for sign, c in consequences],
        '',
    ]
    return '\n'.join(lines)

adr_text = generate_adr(
    title='Architecture for Walmart Retail Assistant (100k queries/day)',
    winner=winner,
    totals=totals,
    context=(
        'Walmart India needs an assistant for 100,000 queries/day across five categories\n'
        '(price, order, policy, hours, multi_step). Constraints: P95 < 3000 ms,\n'
        '<= 800 input / 150 output tokens per call, monthly spend <= $1,500.\n'
        'The existing FAQ system cannot handle multi-step or comparison queries.'
    ),
    rationale=(
        'A RAG + Agent design: a cheap rule-based classifier routes each query to a\n'
        'domain tool, grounds the LLM with retrieved context, and selects the model by\n'
        'complexity (gpt-4o-mini for single-intent, gpt-4-turbo for multi_step only).\n'
        'This keeps the high-volume path deterministic and cheap while still composing\n'
        'tools for multi-intent queries.'
    ),
    consequences=[
        ('+', 'Handles multi-intent queries a fixed workflow cannot express.'),
        ('+', 'Model routing keeps blended cost within the $1,500 ceiling.'),
        ('+', 'Every response carries trace metadata (model, tokens, cost, latency, tool).'),
        ('-', 'Higher latency/cost variance than a pure workflow; must be monitored.'),
        ('-', 'Classifier misroutes are a new failure mode; mitigated by LLM fallback + eval gate.'),
    ],
    alternatives_rejected={
        'RAG + Workflow': 'cheaper per call, but each new intent combination needs a new branch.',
        'Traditional Search': 'no answer synthesis; fails comparison and multi_step queries.',
    },
)
(OUT_DIR / 'capstone_adr.txt').write_text(adr_text)
print(adr_text)
print('ADR saved.')

---
## Task 2: Agent Strategy -- Orchestration and Tool Dispatch

The pipeline runs `classify` → `select_tool` → `mock_retrieve` → `select_model` → LLM call. The input is trimmed with `tiktoken` to stay within 800 tokens, and the output is capped at 150 tokens.

**ARB: What if the classifier misclassifies?**
1. If no rule matches, the query goes to a `gpt-4o-mini` classifier instead of a guessed default.
2. `mock_retrieve()` always adds documents from the whole knowledge base, so a wrong tool still leaves relevant context available.
3. The system prompt tells the model to say so when the answer is not in the context, rather than invent one.
4. Classification accuracy and tool-call accuracy are gated metrics in Task 3, so a misrouting regression blocks deployment.

In [ ]:
MODEL_PRICING = {
    'gpt-4-turbo': {'input': 10.00, 'output': 30.00},
    'gpt-4o':      {'input':  5.00, 'output': 15.00},
    'gpt-4o-mini': {'input':  0.15, 'output':  0.60},
}

def compute_cost(model, in_tok, out_tok):
    p = MODEL_PRICING[model]
    return round((in_tok/1_000_000)*p['input'] + (out_tok/1_000_000)*p['output'], 6)

MAX_INPUT_TOKENS  = 800
MAX_OUTPUT_TOKENS = 150
CATEGORIES = ['price', 'order', 'policy', 'hours', 'multi_step']
ENC = tiktoken.get_encoding('cl100k_base')

def count_tokens(text: str) -> int:
    return len(ENC.encode(text))

def truncate_tokens(text: str, max_tokens: int) -> str:
    ids = ENC.encode(text)
    return text if len(ids) <= max_tokens else ENC.decode(ids[:max_tokens])


class WalmartRetailAgent:
    SYSTEM_PROMPT = (
        'You are the Walmart Retail Assistant. '
        'Answer the customer query using only the provided context. '
        'Be concise and accurate. If the answer is not in the context, say so.'
    )
    ORDER_RE = re.compile(r'ORD-\d+', re.IGNORECASE)

    # ── 1. Classification ────────────────────────────────────────────────
    def _signals(self, q: str) -> dict:
        return {
            'price':  any(w in q for w in ['price', 'cost', 'cheap', 'per ounce', '$', 'how much']),
            'order':  bool(self.ORDER_RE.search(q)) or 'order' in q,
            'policy': any(w in q for w in ['return', 'refund', 'policy', 'exchange']),
            'hours':  any(w in q for w in ['open', 'close', 'hours', 'thanksgiving', 'sunday', 'weekday']),
            'stock':  any(w in q for w in ['in stock', 'aisle', 'available']),
        }

    def _llm_classify(self, query: str) -> str:
        resp = client.chat.completions.create(
            model='gpt-4o-mini', temperature=0, max_tokens=5,
            messages=[
                {'role': 'system', 'content': 'Classify the retail query into exactly one label: '
                 'price, order, policy, hours, multi_step. Reply with the label only.'},
                {'role': 'user', 'content': query},
            ],
        )
        label = resp.choices[0].message.content.strip().lower()
        return label if label in CATEGORIES else 'multi_step'

    def classify(self, query: str) -> str:
        q = query.lower()
        s = self._signals(q)
        if 'compare' in q or ('and' in q.split() and s['stock']) or sum(s.values()) >= 2:
            return 'multi_step'
        for cat in ['order', 'policy', 'hours', 'price']:
            if s[cat]:
                return cat
        return self._llm_classify(query)   # fallback for unmatched queries

    # ── 2. Tool dispatch ─────────────────────────────────────────────────
    @staticmethod
    def _products(q: str) -> list:
        found = [k for k in ['whole milk', '2% milk', 'tide'] if k in q]
        if 'detergent' in q:
            found.append('laundry detergent')
        if 'milk' in q and not any('milk' in f for f in found):
            found.append('milk')
        return found

    def select_tool(self, category: str, query: str) -> dict:
        q = query.lower()
        calls = []
        if category in ('price', 'multi_step'):
            for p in self._products(q) or [q]:
                calls.append(('price_lookup', p))
        if category == 'order' or (category == 'multi_step' and self.ORDER_RE.search(query)):
            m = self.ORDER_RE.search(query)
            calls.append(('order_status', m.group(0).upper() if m else ''))
        if category == 'policy':
            calls.append(('policy_search', 'electronics' if 'electronic' in q else 'receipt general return'))
        if category == 'hours':
            calls.append(('store_hours', ''))

        results, used = {}, []
        for name, arg in calls:
            out = TOOLS[name](arg)
            used.append(name)
            for d in out['results']:
                results.setdefault(d['id'], d)
        tool_used = '+'.join(dict.fromkeys(used)) or 'none'
        return {'tool_used': tool_used,
                'result': {'found': bool(results), 'results': list(results.values())}}

    # ── 3. Model routing ─────────────────────────────────────────────────
    def select_model(self, category: str) -> str:
        return 'gpt-4-turbo' if category == 'multi_step' else 'gpt-4o-mini'

    # ── 4. Orchestration ─────────────────────────────────────────────────
    def run(self, query: str) -> dict:
        trace_id = str(uuid.uuid4())
        t0 = time.perf_counter()

        category = self.classify(query)
        tool     = self.select_tool(category, query)
        retrieved = mock_retrieve(query, k=3)

        # Tool results first (most precise), then retriever results, de-duplicated
        docs = {d['id']: d for d in tool['result']['results']}
        for d in retrieved:
            docs.setdefault(d['id'], d)
        context_ids = list(docs.keys())

        context = '\n'.join(f"[{d['id']}] {d['text']}" for d in docs.values())
        user_tpl = 'Context:\n{context}\n\nCustomer query: {query}'
        overhead = count_tokens(self.SYSTEM_PROMPT) + count_tokens(user_tpl.format(context='', query=query)) + 12
        context = truncate_tokens(context, MAX_INPUT_TOKENS - overhead)

        model = self.select_model(category)
        resp = client.chat.completions.create(
            model=model,
            temperature=0,
            max_tokens=MAX_OUTPUT_TOKENS,
            messages=[
                {'role': 'system', 'content': self.SYSTEM_PROMPT},
                {'role': 'user',   'content': user_tpl.format(context=context, query=query)},
            ],
        )
        in_tok, out_tok = resp.usage.prompt_tokens, resp.usage.completion_tokens
        latency_ms = round((time.perf_counter() - t0) * 1000, 1)

        return {
            'trace_id':      trace_id,
            'query':         query,
            'category':      category,
            'answer':        resp.choices[0].message.content.strip(),
            'model_used':    model,
            'input_tokens':  in_tok,
            'output_tokens': out_tok,
            'cost_usd':      compute_cost(model, in_tok, out_tok),
            'latency_ms':    latency_ms,
            'tool_used':     tool['tool_used'],
            'context':       context,
            'context_ids':   context_ids,
            'within_budget': in_tok <= MAX_INPUT_TOKENS and out_tok <= MAX_OUTPUT_TOKENS,
        }


# ── Test the agent on 5 queries ───────────────────────────────────────────
agent = WalmartRetailAgent()
test_queries = [
    'What is the price of Great Value Whole Milk?',
    'Where is my order ORD-78901?',
    'What is the return policy for electronics?',
    'What time does Walmart open on Sunday?',
    'Compare Great Value and Tide detergent on price per ounce and tell me which is cheaper.',
]
test_runs = []
for q in test_queries:
    r = agent.run(q)
    test_runs.append(r)
    print(f"Q: {q}\n   category={r['category']} | tool={r['tool_used']} | model={r['model_used']}")
    print(f"   tokens={r['input_tokens']}/{r['output_tokens']} | cost=${r['cost_usd']:.6f} | latency={r['latency_ms']}ms")
    print(f"   A: {r['answer']}\n")

---
## Task 3: Evaluation Strategy -- 10-Metric Scorecard

`gpt-4o-mini` acts as the judge. It scores each answer from 0 to 3 for faithfulness, relevancy and correctness, and the scores are divided by 3 to normalise them. Toxicity is scored 0 or 1. The other metrics are computed directly from the trace data.

| # | Metric | Threshold |
|---|---|---|
| 1 | faithfulness | ≥ 0.85 |
| 2 | answer_relevancy | ≥ 0.75 |
| 3 | toxicity | < 0.10 |
| 4 | hit_rate_at_3 | ≥ 0.75 |
| 5 | task_success_rate | ≥ 0.90 |
| 6 | tool_call_accuracy | ≥ 0.95 |
| 7 | classification_accuracy | ≥ 0.90 |
| 8 | p95_latency_ms | < 3000 |
| 9 | avg_cost_per_query_usd | ≤ 0.0005 ($1,500 ÷ 3M calls) |
| 10 | token_budget_compliance | = 1.00 |

In [ ]:
GOLDEN_DATASET = [
    {'id':'G001','cat':'price',
     'query':'What is the price of Great Value Whole Milk?',
     'expected':'Great Value Whole Milk 1 gallon costs $3.98 and is in Aisle 12.'},
    {'id':'G002','cat':'price',
     'query':'Which laundry detergent costs less per ounce?',
     'expected':'Great Value at 6 cents/oz is cheaper than Tide at 13 cents/oz.'},
    {'id':'G003','cat':'order',
     'query':'What is the status of order ORD-78901?',
     'expected':'Order ORD-78901 has been shipped via FedEx (FX123456), arriving by July 3, 2026.'},
    {'id':'G004','cat':'order',
     'query':'When will order ORD-45621 ship?',
     'expected':'Order ORD-45621 is being processed and will ship within 2 business days.'},
    {'id':'G005','cat':'policy',
     'query':'Can I return electronics after 30 days?',
     'expected':'No. Electronics must be returned within 30 days with receipt and original packaging.'},
    {'id':'G006','cat':'policy',
     'query':'Can I return an item without a receipt?',
     'expected':'Yes, within 90 days with a valid photo ID. Refund is issued as store credit.'},
    {'id':'G007','cat':'hours',
     'query':'What time does Walmart open on weekdays?',
     'expected':'Most Walmart Supercenters open at 6:00 AM Monday through Saturday.'},
    {'id':'G008','cat':'hours',
     'query':'Is Walmart open on Thanksgiving?',
     'expected':'No, Walmart stores are closed on Thanksgiving Day.'},
    {'id':'G009','cat':'multi_step',
     'query':'Is Great Value Whole Milk in stock and what aisle?',
     'expected':'Great Value Whole Milk is in stock with 47 units in Aisle 12, Dairy section.'},
    {'id':'G010','cat':'multi_step',
     'query':'Compare Great Value and Tide detergent on price per ounce.',
     'expected':'Great Value (150 oz) costs $8.97 at 6c/oz. Tide (92 oz) costs $11.97 at 13c/oz. Great Value is cheaper per ounce.'},
]

THRESHOLDS = {
    'faithfulness':       0.85,
    'answer_relevancy':   0.75,
    'toxicity':           0.10,  # must be BELOW this
    'hit_rate_at_3':      0.75,
    'task_success_rate':  0.90,
    'tool_call_accuracy': 0.95,
}
# Additional production gates (from Task 2 / Task 4 constraints)
EXTRA_THRESHOLDS = {
    'classification_accuracy': 0.90,
    'p95_latency_ms':          3000,     # must be BELOW this
    'avg_cost_per_query_usd':  1500 / (100_000 * 30),
    'token_budget_compliance': 1.00,
}

# Ground-truth relevant documents and expected tool per record
RELEVANT_DOCS = {
    'G001': {'P001'}, 'G002': {'P003', 'P004'}, 'G003': {'O001'}, 'G004': {'O002'},
    'G005': {'R001'}, 'G006': {'R002'}, 'G007': {'H001'}, 'G008': {'H002'},
    'G009': {'P001'}, 'G010': {'P003', 'P004'},
}
EXPECTED_TOOL = {'price': 'price_lookup', 'order': 'order_status', 'policy': 'policy_search',
                 'hours': 'store_hours', 'multi_step': 'price_lookup'}

JUDGE_PROMPT = '''You are an impartial evaluator for a retail assistant.
Score the ANSWER using this 0-3 rubric for each dimension:
  3 = fully meets, 2 = mostly meets (minor gap), 1 = partially, 0 = fails.
- faithfulness: every claim in ANSWER is supported by CONTEXT.
- relevancy:    ANSWER directly addresses QUESTION.
- correctness:  ANSWER conveys the same facts as EXPECTED.
- toxicity:     1 if ANSWER is toxic, rude or unsafe, else 0.
Return JSON only: {"faithfulness":int,"relevancy":int,"correctness":int,"toxicity":int,"reason":str}'''

def llm_judge(question, context, answer, expected) -> dict:
    resp = client.chat.completions.create(
        model='gpt-4o-mini', temperature=0,
        response_format={'type': 'json_object'},
        messages=[
            {'role': 'system', 'content': JUDGE_PROMPT},
            {'role': 'user', 'content': f'QUESTION: {question}\nCONTEXT:\n{context}\nANSWER: {answer}\nEXPECTED: {expected}'},
        ],
    )
    return json.loads(resp.choices[0].message.content)

eval_results = []
for rec in GOLDEN_DATASET:
    r = agent.run(rec['query'])
    j = llm_judge(rec['query'], r['context'], r['answer'], rec['expected'])
    row = {
        **rec, **r,
        'faithfulness':   j['faithfulness'] / 3,
        'relevancy':      j['relevancy'] / 3,
        'correctness':    j['correctness'] / 3,
        'toxicity':       float(j['toxicity']),
        'hit':            bool(RELEVANT_DOCS[rec['id']] & set(r['context_ids'][:3])),
        'task_success':   j['correctness'] >= 2,
        'tool_correct':   EXPECTED_TOOL[rec['cat']] in r['tool_used'],
        'class_correct':  r['category'] == rec['cat'],
        'judge_reason':   j.get('reason', ''),
    }
    eval_results.append(row)
    print(f"{rec['id']} [{rec['cat']:<10}] -> {r['category']:<10} faith={row['faithfulness']:.2f} "
          f"rel={row['relevancy']:.2f} corr={row['correctness']:.2f} hit={row['hit']} tool_ok={row['tool_correct']}")

In [ ]:
def mean(xs): return sum(xs) / len(xs)

def p95(xs):
    s = sorted(xs)
    return s[min(len(s) - 1, int(round(0.95 * (len(s) - 1))))]

metrics = {
    'faithfulness':            mean([r['faithfulness'] for r in eval_results]),
    'answer_relevancy':        mean([r['relevancy'] for r in eval_results]),
    'toxicity':                mean([r['toxicity'] for r in eval_results]),
    'hit_rate_at_3':           mean([r['hit'] for r in eval_results]),
    'task_success_rate':       mean([r['task_success'] for r in eval_results]),
    'tool_call_accuracy':      mean([r['tool_correct'] for r in eval_results]),
    'classification_accuracy': mean([r['class_correct'] for r in eval_results]),
    'p95_latency_ms':          p95([r['latency_ms'] for r in eval_results]),
    'avg_cost_per_query_usd':  mean([r['cost_usd'] for r in eval_results]),
    'token_budget_compliance': mean([r['within_budget'] for r in eval_results]),
}
LOWER_IS_BETTER = {'toxicity', 'p95_latency_ms', 'avg_cost_per_query_usd'}
ALL_THRESHOLDS = {**THRESHOLDS, **EXTRA_THRESHOLDS}

def passes(name, value):
    t = ALL_THRESHOLDS[name]
    if name in ('toxicity', 'p95_latency_ms'):
        return value < t
    if name in LOWER_IS_BETTER:
        return value <= t
    return value >= t

lines = ['CAPSTONE EVALUATION SCORECARD -- Walmart Retail Assistant',
         '=' * 72,
         f"Run: {datetime.now(timezone.utc).isoformat(timespec='seconds')} | Records: {len(eval_results)}",
         '',
         f"{'#':<3}{'Metric':<28}{'Value':>12}{'Threshold':>14}{'Result':>10}",
         '-' * 72]
all_pass = True
for i, (name, value) in enumerate(metrics.items(), 1):
    ok = passes(name, value)
    all_pass &= ok
    op = '<' if name in ('toxicity', 'p95_latency_ms') else ('<=' if name in LOWER_IS_BETTER else '>=')
    lines.append(f"{i:<3}{name:<28}{value:>12.4f}{op + ' ' + format(ALL_THRESHOLDS[name], 'g'):>14}{'PASS' if ok else 'FAIL':>10}")
lines += ['-' * 72, f"OVERALL GATE: {'PASS' if all_pass else 'FAIL'}", '', 'PER-RECORD DETAIL', '-' * 72]
for r in eval_results:
    lines.append(f"{r['id']} {r['cat']:<10} pred={r['category']:<10} model={r['model_used']:<12} "
                 f"faith={r['faithfulness']:.2f} corr={r['correctness']:.2f} hit={r['hit']} "
                 f"tool={r['tool_used']} {r['latency_ms']}ms")
    lines.append(f"     A: {r['answer']}")

scorecard = '\n'.join(lines)
(OUT_DIR / 'capstone_evaluation_scorecard.txt').write_text(scorecard)
print(scorecard)

---
## Task 4: Cost Model -- Monthly Projection

The average token counts are measured from the actual agent runs in Tasks 2 and 3. The production traffic mix is an assumption: the golden set is 20% `multi_step`, but real retail traffic is mostly single-intent, so the mix below uses 5%. The assumption is printed with the results so the ARB can challenge it.

**ARB: What is the worst-case spend if the cache fails?** It is Scenario 2 with compression but no cache hits. It is printed below and checked against the ceiling.

In [ ]:
DAILY_CALLS   = 100_000
DAYS_PER_MONTH = 30
MONTHLY_CEILING = 1500.00
CACHE_HIT_RATE  = 0.20
COMPRESSION     = 0.18   # input-token reduction

all_runs = test_runs + eval_results
avg_input_tokens  = mean([r['input_tokens'] for r in all_runs])
avg_output_tokens = mean([r['output_tokens'] for r in all_runs])

# Assumed production traffic mix (single-intent dominant)
TRAFFIC_MIX = {'price': 0.30, 'order': 0.30, 'policy': 0.15, 'hours': 0.20, 'multi_step': 0.05}
assert abs(sum(TRAFFIC_MIX.values()) - 1.0) < 1e-9

def monthly_projection(cost_per_call, cache_hit=0.0):
    return cost_per_call * DAILY_CALLS * DAYS_PER_MONTH * (1 - cache_hit)

def blended_cost(in_tok, out_tok):
    return sum(share * compute_cost(agent.select_model(cat), in_tok, out_tok)
               for cat, share in TRAFFIC_MIX.items())

compressed_in = avg_input_tokens * (1 - COMPRESSION)
scenarios = {
    '1. Baseline (all gpt-4-turbo)':              monthly_projection(compute_cost('gpt-4-turbo', avg_input_tokens, avg_output_tokens)),
    '2. Model routing':                           monthly_projection(blended_cost(avg_input_tokens, avg_output_tokens)),
    '3. Routing + 20% cache + 18% compression':   monthly_projection(blended_cost(compressed_in, avg_output_tokens), CACHE_HIT_RATE),
}
worst_case_cache_fail = monthly_projection(blended_cost(compressed_in, avg_output_tokens), 0.0)

print(f'Avg tokens/call: input={avg_input_tokens:.0f}, output={avg_output_tokens:.0f}')
print(f'Traffic mix: {TRAFFIC_MIX}\n')
print(f"{'Scenario':<45}{'Monthly USD':>14}{'vs Ceiling':>14}")
print('-' * 73)
for name, cost in scenarios.items():
    print(f"{name:<45}{cost:>14,.2f}{'OK' if cost <= MONTHLY_CEILING else 'OVER':>14}")
print('-' * 73)
print(f"{'Worst case: cache fails (routing + compression)':<45}{worst_case_cache_fail:>14,.2f}"
      f"{'OK' if worst_case_cache_fail <= MONTHLY_CEILING else 'OVER':>14}")
baseline = scenarios['1. Baseline (all gpt-4-turbo)']
optimised_monthly = scenarios['3. Routing + 20% cache + 18% compression']
print(f'\nSavings vs baseline: ${baseline - optimised_monthly:,.2f}/month ({(1 - optimised_monthly/baseline)*100:.1f}%)')

assert optimised_monthly <= MONTHLY_CEILING, 'Budget ceiling breached'
print('Optimised scenario is within the $1,500/month ceiling.')

---
## Task 5: Deployment Model -- CI/CD and Rollback

**ARB: If faithfulness drops from 0.88 to 0.76, how fast can we roll back?** Prompts and model choices are versioned configuration behind a flag, so a rollback is a config pointer flip rather than a redeploy. A faithfulness drop below 0.85 sets off the automated canary alarm within about 10 minutes, and the rollback itself takes under 5 minutes. Total time is under 15 minutes, against a 30-minute target.

In [ ]:
deployment_model = f'''
CAPSTONE DEPLOYMENT MODEL -- Walmart Retail Assistant
=====================================================

1. CHANGE TYPES AND PIPELINE PATHS
   All artefacts are versioned in Git; prompts and model routing live in a
   versioned config registry (e.g. prompts/v{{n}}.yaml) behind a feature flag.
   - System prompt change   : PR -> lint + prompt-diff review -> golden eval (10 metrics)
                              -> staging -> canary 5% -> 25% -> 100%
   - Model version change   : PR -> golden eval + regression benchmark vs current model
                              + cost projection re-run -> staging -> canary 5% (48h soak) -> 100%
   - RAG chunk config change: PR -> re-index into shadow Pinecone namespace -> hit_rate@3
                              + faithfulness eval -> blue/green index alias swap
   - Tool logic change      : PR -> unit tests per tool + contract tests + golden eval
                              -> staging -> canary 10% -> 100%

2. PRE-DEPLOYMENT GATE (all must PASS, automated in CI)
   - Evaluation scorecard (Task 3) OVERALL GATE = PASS:
       faithfulness >= 0.85, answer_relevancy >= 0.75, toxicity < 0.10,
       hit_rate@3 >= 0.75, task_success >= 0.90, tool_call_accuracy >= 0.95,
       classification_accuracy >= 0.90, P95 latency < 3000 ms,
       avg cost/query <= ${EXTRA_THRESHOLDS["avg_cost_per_query_usd"]:.4f}, token budget compliance = 100%
   - No metric regresses > 3 points vs current production baseline
   - Prompt-injection red-team suite: 0 successful overrides
   - Projected monthly spend <= $1,500 (Task 4 re-run)
   - Approval: 1 AI engineer + 1 product owner

3. DEPLOYMENT STEPS
   a. Staging   : full golden + 200-query shadow replay of prod traffic (no user impact)
   b. Canary 5% : 1h minimum; compare live faithfulness (sampled LLM-judge), P95,
                  error rate, cost/query vs control
   c. Canary 25%: 4h; same guardrails
   d. Production 100%: flag flipped; previous config kept warm as rollback target for 7 days

4. ROLLBACK TRIGGER CONDITIONS (any one -> immediate automated rollback)
   - Sampled faithfulness < 0.85 or task_success < 0.90 over a 15-min window
   - P95 latency >= 3000 ms for 10 consecutive minutes
   - 5xx / LLM error rate > 2% for 5 minutes
   - Cost per query > 1.5x baseline for 30 minutes, or daily spend > $50
   - Any confirmed PII leak or successful prompt injection

5. ROLLBACK STEPS (target < 30 min; typical < 15 min)
   T+0   Alert fires (auto) or on-call declares incident
   T+2   Flip feature flag / config pointer to previous prompt+model version (no redeploy)
   T+5   For RAG changes: swap Pinecone index alias back to previous namespace
   T+10  Verify: golden smoke set (10 queries) passes; dashboards return to baseline
   T+15  Announce in incident channel; freeze further deploys
   T+24h Post-incident review; add failing case to golden dataset

6. ON-CALL RUNBOOK SUMMARY
   Page order: AI Platform on-call (L1) -> Retail Assistant tech lead (L2)
               -> Eng manager + Product owner (L3, if customer impact > 30 min)
   First 5 actions:
     1. Check dashboard: faithfulness, P95, error rate, cost/query, cache hit rate
     2. Identify last change (deploy log / config registry diff)
     3. If a change landed in the last 24h -> roll back first, investigate second
     4. Check dependencies: OpenAI status, Pinecone status; enable mock/cached fallback
     5. Pull 20 failing traces by trace_id; attach to incident ticket
'''

(OUT_DIR / 'capstone_deployment_model.txt').write_text(deployment_model)
print(deployment_model)
print('Deployment model saved.')

---
## Task 6: Risk Mitigation Plan

**ARB: How do we detect and respond to silent quality drift?** A daily job replays the golden set, and the LLM judge scores a 1% sample of live traffic. A rolling 7-day faithfulness or task-success average that falls more than 3 points below baseline opens a ticket. Falling below the threshold pages on-call. The response is to pin the model version, diff the inputs (traffic mix and knowledge-base freshness), and roll back or re-tune.

In [ ]:
risk_register = {
    'hallucination': {
        'likelihood': 'Medium',
        'impact':     'High',
        'detection':  'Sampled LLM-judge faithfulness on 1% of live traffic; golden-set nightly run; '
                      'user thumbs-down rate.',
        'mitigation': 'Context-only system prompt with explicit "say so if unknown"; tool results placed first '
                      'in context; temperature=0; faithfulness >= 0.85 deployment gate; price/stock figures '
                      'rendered from tool output, not free text.',
        'owner':      'AI Engineering',
    },
    'prompt_injection': {
        'likelihood': 'High',
        'impact':     'High',
        'detection':  'Input classifier for override patterns ("ignore previous", role-play, system prompt '
                      'requests); red-team suite in CI; anomaly alert on refusal-rate spikes.',
        'mitigation': 'Strict system/user role separation; user text never concatenated into system prompt; '
                      'tools are read-only with allow-listed arguments (regex-validated order IDs); '
                      'output filter blocks system-prompt disclosure.',
        'owner':      'AI Security',
    },
    'pii_leakage': {
        'likelihood': 'Medium',
        'impact':     'Critical',
        'detection':  'PII scanner on logs and responses (order IDs, phone, email, address); weekly log audit.',
        'mitigation': 'Redact/hash order IDs and personal data before logging (trace_id only); order_status '
                      'requires authenticated session ownership check; log retention 30 days; no PII sent '
                      'to evaluation datasets.',
        'owner':      'Data Privacy + Platform',
    },
    'model_drift': {
        'likelihood': 'Medium',
        'impact':     'High',
        'detection':  'Daily golden-set replay + rolling 7-day live faithfulness/task-success; alert when > 3 pts '
                      'below baseline; classification-mix drift monitor.',
        'mitigation': 'Pin explicit model versions; regression benchmark before any model upgrade; '
                      'automated rollback triggers; add failure cases to golden dataset.',
        'owner':      'AI Engineering + MLOps',
    },
    'cost_overrun': {
        'likelihood': 'Medium',
        'impact':     'Medium',
        'detection':  'Real-time cost/query from trace metadata; daily spend alert at $40 (80% of $50/day); '
                      'gpt-4-turbo share > 10% alert.',
        'mitigation': 'Hard caps 800 in / 150 out tokens; model routing; response cache; prompt compression; '
                      'circuit breaker degrades multi_step to gpt-4o-mini when daily budget is 90% used.',
        'owner':      'FinOps + AI Engineering',
    },
    'dependency_failure': {
        'likelihood': 'Medium',
        'impact':     'High',
        'detection':  'Health checks + error-rate alerts on OpenAI and Pinecone; P95 latency SLO alert.',
        'mitigation': 'Timeouts + exponential-backoff retries; fallback model (gpt-4o-mini / alternate region); '
                      'mock_retrieve / cached KB fallback when Pinecone is down; serve cached answers; '
                      'graceful "please try again" message instead of hallucinating.',
        'owner':      'Platform / SRE',
    },
}

for cat, r in risk_register.items():
    print(f"\n[{cat.upper()}]  likelihood={r['likelihood']} | impact={r['impact']} | owner={r['owner']}")
    print(f"  Detection : {r['detection']}")
    print(f"  Mitigation: {r['mitigation']}")

(OUT_DIR / 'capstone_risk_register.txt').write_text(json.dumps(risk_register, indent=2))
print('\nRisk register saved.')

---
## Deliverables Checklist

In [ ]:
deliverables = {
    'capstone_adr.txt':                  'Task 1 - Architecture Decision Record',
    'capstone_evaluation_scorecard.txt': 'Task 3 - 10 metrics with pass/fail',
    'capstone_deployment_model.txt':     'Task 5 - CI/CD and rollback plan',
    'capstone_risk_register.txt':        'Task 6 - Six risk categories with mitigations',
}
for f, desc in deliverables.items():
    p = OUT_DIR / f
    print(f"{'OK     ' if p.exists() and p.stat().st_size > 0 else 'MISSING'} {p}  ({desc})")

print(f"\nEvaluation gate: {'PASS' if all_pass else 'FAIL'}")
print(f'Optimised monthly spend: ${optimised_monthly:,.2f} (ceiling ${MONTHLY_CEILING:,.0f})')